# UniBarCode — Python quickstart

`unibarcode` is a Cython extension over the UniBarCode C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unibarcode
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import unibarcode

unibarcode.version(), unibarcode.__version__

('1.0.0', '1.0.0')

Encode a payload by symbology name (or ordinal). The result carries module dimensions and render methods.

In [2]:
bc = unibarcode.encode("ean13", "978020137962")
bc.is_ok, bc.width, bc.height, bc.is_2d

(True, 95, 1, False)

2-D symbologies produce a grid.

In [3]:
qr = unibarcode.encode("qr", "Hello")
qr.is_2d, qr.width, qr.height

(True, 21, 21)

## Validation

A bad payload returns a result with `is_ok == False` and an error message,
rather than raising — the caller decides how to handle it.

In [4]:
bad = unibarcode.encode("ean13", "abc")
bad.is_ok, bad.error

(False, 'validation error')

An unknown symbology raises immediately.

In [5]:
try:
    unibarcode.encode("nope", "x")
except ValueError as exc:
    print("ValueError:", exc)

ValueError: unknown symbology: 'nope'


## Render

`render_png` and `render_svg` both return bytes -- the SVG is a standalone document, not a `str`. Options control module size, bar height, colors, and the HRI text strip.

In [6]:
png = bc.render_png()
png[:4]

b'\x89PNG'

In [7]:
opts = unibarcode.Options().module_size(3).bar_height(120).show_hri(False)
svg = bc.render_svg(opts)
svg[:20]

b'<svg xmlns="http://w'

Colors parse any CSS Color 4 string.

In [8]:
opts = unibarcode.Options().foreground(unibarcode.Color.parse("#3366cc"))
bc.render_png(opts)[:4]

b'\x89PNG'

## The C ABI underneath

The same entry points are reachable from anything that speaks C. No exception
ever unwinds across the ABI boundary — failures map to `UBC_*` status codes:

```c
ubc_barcode *h = ubc_encode(UBC_SBC_EAN13, "978020137962");
ubc_render_png(h, NULL, &out, &len);   /* UBC_OK */
```

See `include/UniBarCode.h`, and the book for the full picture.